In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [ ]:
nav = pd.read_csv("data/raw/02_nav_history.csv")

transactions = pd.read_csv(
    "data/raw/08_investor_transactions.csv"
)

performance = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)

fund_master = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

In [ ]:
print(nav.shape)
print(nav.head())

print(nav.info())

In [ ]:
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

In [ ]:
nav["date"].dtype

In [ ]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [ ]:
nav["nav"] = (
    nav
    .groupby("amfi_code")["nav"]
    .ffill()
)

In [ ]:
nav = nav.drop_duplicates()

In [ ]:
nav = nav.drop_duplicates(
    subset=["amfi_code","date"]
)

In [ ]:
invalid_nav = nav[nav["nav"]<=0]

In [ ]:
nav = nav[nav["nav"]>0]

In [ ]:
nav.to_csv(
    "data/processed/nav_history.csv",
    index=False
)

In [ ]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.str.strip()
.str.lower()
)

In [ ]:
mapping = {

"sip":"SIP",

"systematic investment":"SIP",

"lumpsum":"Lumpsum",

"lump sum":"Lumpsum",

"redemption":"Redemption"

}

In [ ]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.replace(mapping)
)

In [ ]:
transactions = transactions[
transactions["amount_inr"]>0
]

In [ ]:
transactions["transaction_date"] = pd.to_datetime(

transactions["transaction_date"],

errors="coerce"
)

In [ ]:
valid = [
"Verified",
"Pending",
"Rejected"
]

invalid = transactions[
~transactions["kyc_status"].isin(valid)
]

In [ ]:
transactions.to_csv(
"data/processed/investor_transactions.csv",
index=False
)

In [ ]:
cols = [
"return_1yr_pct",
"return_3yr_pct",
"return_5yr_pct"
]

for c in cols:

    performance[c]=pd.to_numeric(
        performance[c],
        errors="coerce"
    )

In [ ]:
performance[
performance["return_1yr_pct"]>200
]

In [ ]:
performance[
performance["return_1yr_pct"]<-100
]

In [ ]:
performance[
(performance["expense_ratio_pct"]<0.1)
|
(performance["expense_ratio_pct"]>2.5)
]

In [ ]:
performance.to_csv(
"data/processed/scheme_performance.csv",
index=False
)

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///database/bluestock_mf.db")

In [ ]:
import os

print(os.path.exists("database/bluestock_mf.db"))